# UEBA Portable — Entraînement **per-user** sur Google Colab

Ce notebook entraîne un **modèle d'anomalies dédié par utilisateur**
(`PerUserAnomalyEnsemble` : IsolationForest + OneClassSVM + Autoencoder MLP
pour *chaque* utilisateur), conformément au principe fondamental de l'UEBA
personnalisée (Salem & Stolfo 2011 ; Veeramachaneni et al. 2016).

**Scénario évalué** : Password Spraying — MITRE ATT&CK **T1110.003**,
injecté les 13 et 16 mai 2026 sur plusieurs comptes ciblés.

Les jours d'attaque sont **exclus de l'apprentissage** (baseline propre) puis
**conservés au scoring** pour mesurer le rappel de détection.

> CPU suffisant (16 features) — aucun GPU requis.

## 0. Installation

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Remplacez USER par votre compte/organisation GitHub.
    # Tant que la PR n'est pas fusionnée, ciblez la branche du refacto :
    !pip install -q "git+https://github.com/USER/ueba-portable.git@refactor/per-user-ensemble"
else:
    # Exécution locale depuis notebooks/ : on pointe vers le code source du dépôt.
    sys.path.insert(0, "../src")

print("Environnement :", "Google Colab" if IN_COLAB else "Local")

## 1. Upload du dataset CSV

Sur Colab, déposez votre export Wazuh natif (schéma `data.win.eventdata.*`).
En local, le notebook retombe sur la fixture synthétique du dépôt.

In [ ]:
if IN_COLAB:
    from google.colab import files

    uploaded = files.upload()
    CSV_PATH = next(iter(uploaded))
else:
    CSV_PATH = "../tests/integration/fixtures/sample_logs.csv"

print("Dataset :", CSV_PATH)

## 2. Normalisation via `WazuhAdapter`

In [ ]:
import csv
from pathlib import Path

from ueba.adapters.wazuh import WazuhAdapter

with Path(CSV_PATH).open(newline="", encoding="utf-8") as f:
    records = list(csv.DictReader(f))

events = WazuhAdapter().normalize(records)

dates = sorted({e.timestamp.date().isoformat() for e in events})
users = sorted({e.user for e in events})
print(f"Événements normalisés : {len(events)}")
print(f"Période               : {dates[0]} → {dates[-1]} ({len(dates)} jours)")
print(f"Utilisateurs          : {len(users)} → {users}")

## 3. Extraction des features (fenêtres glissantes)

In [ ]:
from collections import Counter
from datetime import timedelta

from ueba.domain.features import UEBAFeatureExtractor

extractor = UEBAFeatureExtractor(
    window_size=timedelta(hours=1),
    window_step=timedelta(minutes=30),
)
vectors = extractor.extract(events)

per_user = Counter(v.user for v in vectors)
print(f"Vecteurs (utilisateur × fenêtre) : {len(vectors)}")
print("Fenêtres par utilisateur :")
for user, n in sorted(per_user.items(), key=lambda kv: -kv[1]):
    print(f"  {user:20s} {n:4d}")

## 4. Vérité terrain de l'attaque connue

Adaptez ces constantes à votre jeu de données. Les valeurs par défaut
correspondent au scénario de password spraying T1110.003 du projet.

In [ ]:
ATTACK_DATES = ["2026-05-13", "2026-05-16"]
ATTACK_USERS = {
    "a.amrani",
    "l.idrissi",
    "l.mus",
    "y.ben",
    "n.alam",
    "s.ed",
    "k.alaa",
}
print("Jours d'attaque :", ATTACK_DATES)
print("Comptes ciblés  :", sorted(ATTACK_USERS))

## 5. Préparation du jeu d'apprentissage propre, puis entraînement

**Séparation des responsabilités** : `PerUserAnomalyEnsemble` apprend la
normalité de ce qu'on lui fournit ; c'est à l'utilisateur de lui passer un
jeu d'apprentissage *propre*. On retire donc ici, **en amont de la classe**,
les jours connus comme contaminés (`ATTACK_DATES`) pour constituer la
baseline. L'évaluation, elle, se fera sur l'intégralité des fenêtres.

In [ ]:
from ueba.domain.per_user_ensemble import PerUserAnomalyEnsemble

# Jeu d'apprentissage propre : on exclut les jours d'attaque EN AMONT de la classe.
train_vectors = [
    v for v in vectors if v.window_start.date().isoformat() not in ATTACK_DATES
]
print(f"Fenêtres totales         : {len(vectors)}")
print(f"Fenêtres d'apprentissage : {len(train_vectors)} (jours d'attaque exclus)")

# train_ratio=1.0 : on apprend le "normal" sur l'INTÉGRALITÉ du jeu propre
# (cas d'usage SOC : constitution d'une baseline complète, pas d'évaluation).
model = PerUserAnomalyEnsemble(
    min_windows_per_user=30,
    train_ratio=1.0,
    n_estimators=200,
    majority_threshold=2,
    random_state=42,
)
model.fit(train_vectors)

print(f"Modèles entraînés : {len(model.trained_users)}")
print("Utilisateurs      :", model.trained_users)

## 6. Prédiction et évaluation

* **Recall fenêtre par fenêtre** : part des fenêtres d'attaque flaguées.
* **Recall opérationnel** : part des couples (utilisateur × jour d'attaque)
  pour lesquels le SOC reçoit **au moins une** alerte — métrique de référence
  en littérature SOC (Bhatt et al. 2014).
* **Détection en première fenêtre** : la toute première fenêtre de chaque
  couple d'attaque est-elle déjà signalée ?
* **Taux de faux positifs** : alertes sur les jours propres (utilisateurs
  entraînés).

In [ ]:
import pandas as pd

verdicts = model.predict(vectors)

df = pd.DataFrame(
    {
        "user": [v.user for v in vectors],
        "date": [v.window_start.date().isoformat() for v in vectors],
        "window_start": [v.window_start for v in vectors],
        "failed_login_count": [v.failed_login_count for v in vectors],
        "is_anomaly": [verd.is_anomaly for verd in verdicts],
        "was_in_training": [verd.was_in_training for verd in verdicts],
        "used_model": [verd.used_model for verd in verdicts],
    }
)
df["is_attack_user"] = df["user"].isin(ATTACK_USERS)
df["is_attack_date"] = df["date"].isin(ATTACK_DATES)
df["is_attack_window"] = df["is_attack_user"] & df["is_attack_date"]

attack = df[df["is_attack_window"]]
clean = df[(~df["is_attack_date"]) & df["was_in_training"]]

window_recall = attack["is_anomaly"].mean()
op_recall = attack.groupby(["user", "date"])["is_anomaly"].any().mean()
first = attack.sort_values("window_start").groupby(["user", "date"]).first()
detect_first = int(first["is_anomaly"].sum())
n_pairs = len(first)
fp_rate = clean["is_anomaly"].mean()
snr = (window_recall / fp_rate) if fp_rate else float("inf")

print(f"Fenêtres d'attaque évaluées      : {len(attack)}")
print(f"Recall fenêtre par fenêtre       : {window_recall:6.1%}")
print(f"Recall opérationnel (user×jour)  : {op_recall:6.1%}")
print(f"Détection en première fenêtre    : {detect_first}/{n_pairs}")
print(f"FP rate (jours propres)          : {fp_rate:6.1%}")
print(f"Ratio signal / bruit             : {snr:6.2f}")

## 7. Visualisations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

users_axis = sorted(df["user"].unique())
u_index = {u: i for i, u in enumerate(users_axis)}

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 14))

# (a) Timeline des détections
normal = df[~df["is_anomaly"]]
flagged = df[df["is_anomaly"]]
ax1.scatter(normal["window_start"], normal["user"], s=14, c="#aec7e8", label="normal")
ax1.scatter(flagged["window_start"], flagged["user"], s=24, c="#d62728", label="anomalie")
for d in ATTACK_DATES:
    ax1.axvspan(
        np.datetime64(f"{d}T00:00:00"),
        np.datetime64(f"{d}T23:59:59"),
        color="orange",
        alpha=0.12,
    )
ax1.set_title("Timeline des détections (zones orange = jours d'attaque)")
ax1.legend(loc="upper right")
ax1.tick_params(axis="x", rotation=30)

# (b) Distribution du taux d'anomalies par utilisateur
rate = df.groupby("user")["is_anomaly"].mean().reindex(users_axis)
colors = ["#d62728" if u in ATTACK_USERS else "#1f77b4" for u in users_axis]
ax2.bar(users_axis, rate.values, color=colors)
ax2.set_title("Taux d'anomalies par utilisateur (rouge = compte ciblé)")
ax2.set_ylabel("part de fenêtres anormales")
ax2.tick_params(axis="x", rotation=45)

# (c) Heatmap fenêtres (par jour) × utilisateurs
pivot = (
    df.assign(anom=df["is_anomaly"].astype(float))
    .pivot_table(index="user", columns="date", values="anom", aggfunc="mean")
    .reindex(users_axis)
)
im = ax3.imshow(pivot.values, aspect="auto", cmap="Reds", vmin=0, vmax=1)
ax3.set_xticks(range(len(pivot.columns)))
ax3.set_xticklabels(pivot.columns, rotation=45, ha="right")
ax3.set_yticks(range(len(pivot.index)))
ax3.set_yticklabels(pivot.index)
ax3.set_title("Heatmap du taux d'anomalies (utilisateur × jour)")
fig.colorbar(im, ax=ax3, fraction=0.025)

plt.tight_layout()
plt.show()

## 8. Sauvegarde et téléchargement du modèle

In [ ]:
MODEL_PATH = "per_user_ensemble.joblib"
model.save(MODEL_PATH)

# Vérification : le rechargement restaure l'état entraîné et des verdicts identiques.
reloaded = PerUserAnomalyEnsemble.load(MODEL_PATH)
assert reloaded.is_fitted
before = [v.is_anomaly for v in verdicts]
after = [v.is_anomaly for v in reloaded.predict(vectors)]
assert before == after, "Incohérence après rechargement"
print(f"Modèle sauvegardé et rechargé : {MODEL_PATH} (verdicts identiques)")

if IN_COLAB:
    from google.colab import files

    files.download(MODEL_PATH)